# Milestone 1 — Colab CPU
Notebook chung cho smoke tối đa 10 ID và chạy toàn range có cờ bật riêng. Code lấy từ GitHub; dữ liệu nằm dưới Drive root đã chọn. Sau git pull, restart runtime nếu crawler đã import, rồi mount Drive và chạy lại các cell thiết lập.


In [ ]:
from pathlib import Path
import os, subprocess, sys
selected = Path(input('Thư mục dự án đã chọn trên Google Drive: ')).expanduser()
assert selected.is_dir(), 'Chọn thư mục Drive đã mount và tồn tại'
assert input('Xác nhận thư mục này nằm trên Drive (yes): ') == 'yes'
os.chdir(selected)
print('Project root selected')

In [ ]:
repo = Path('code')
if not repo.exists():
    subprocess.run(['git','clone','https://github.com/NewB-S2deeC0de/arXiv.git',str(repo)],check=True)
else:
    subprocess.run(['git','-C',str(repo),'pull','--ff-only'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-r',str(repo/'src/requirements.txt')],check=True)
sys.path.insert(0,str(repo/'src'))

In [ ]:
from crawler.config import load_config
from crawler.pipeline import run_pipeline
from crawler.secrets import load_semantic_scholar_key
config = load_config(Path('code/config/crawler.yaml'))
ALLOW_ANONYMOUS_SMOKE = False  # Chỉ bật nếu chủ động thử smoke không key.
config.execution['allow_anonymous_s2'] = ALLOW_ANONYMOUS_SMOKE
print('authentication=' + ('key' if load_semantic_scholar_key() else 'anonymous'))
print('Discovery batch:', config.network['discovery_batch_size'], 'Version metadata batch:', config.network['metadata_batch_size'])

from crawler.ids import generate_ids
input_ids = generate_ids(**config.range)
print('Range inclusive:', input_ids[0], '→', input_ids[-1], '| Tổng:', len(input_ids))


## Scratch SSD tùy chọn
Chạy sau cell cấu hình để tải/giải nén trên SSD; result và progress vẫn lưu tại persistent_root trên Drive. Symlink hỏng sau runtime reset được tạo lại. Để False nếu muốn giữ scratch_root trong YAML.


In [ ]:
USE_LOCAL_SCRATCH = True
if USE_LOCAL_SCRATCH:
    import tempfile
    scratch_dir = Path('scratch')
    if scratch_dir.is_symlink() and not scratch_dir.exists():
        scratch_dir.unlink()
    if not scratch_dir.exists():
        scratch_dir.symlink_to(tempfile.mkdtemp(prefix='arxiv-crawler-'), target_is_directory=True)
    assert scratch_dir.is_dir()
    config.paths['scratch_root'] = 'scratch'
    config.validate()
print('Scratch project path:', config.paths['scratch_root'])


## Smoke: tối đa 10 ID
Cell này gọi mạng thật khi bạn chủ động chạy. Dữ liệu nằm trong runtime/ tương đối với Drive root đã chọn.

In [ ]:
summary = await run_pipeline(config, {'arxiv','s2'}, resume=True, limit=10)
print(summary)

## Status và resume
Ngắt cell chạy bằng nút Stop, sau đó chạy lại cell smoke. Chỉ đặt takeover_stale=True nếu runtime trước đã dừng hoàn toàn.

In [ ]:
from crawler.progress import ProgressStore
store = ProgressStore(config.paths['persistent_root'])
print(store.status_counts())


## Khi nhận API key
Thêm SEMANTIC_SCHOLAR_API_KEY trong Colab Secrets, bật quyền notebook. Chạy lại cell kiểm tra authentication, rồi cell smoke. Các stage failed chỉ được thử lại khi chủ động chạy cell sau; không reset dữ liệu thành công.

In [ ]:
from crawler.diagnostics import diagnose_sources
import json
print(json.dumps(diagnose_sources(config), indent=2, ensure_ascii=False))


In [ ]:
# Chỉ bật sau khi đã xem diagnosis và chắc chắn không có phiên khác đang chạy.
RETRY_FAILED_SOURCES = False
if RETRY_FAILED_SOURCES:
    from crawler.locking import DatasetLock
    from crawler.ids import generate_ids
    import uuid
    with DatasetLock(config.paths['persistent_root'], 'retry-' + uuid.uuid4().hex):
        store = ProgressStore(config.paths['persistent_root'])
        store.reset('failed', stages={'source'}, ids=set(generate_ids(**config.range)[:10]))
    summary = await run_pipeline(config, {'arxiv'}, resume=True, limit=10)
    print(summary)


## Chạy / resume toàn range
Chỉ bật RUN_FULL_RANGE sau smoke và interrupt/resume đã đạt. Cell dùng limit=None, nghĩa là toàn bộ config.range; không cố định 5.000 nếu bạn đổi range. Dùng đúng persistent_root cũ để resume. Cell này yêu cầu S2 API key và không reset failed jobs.


In [ ]:
RUN_FULL_RANGE = False  # Đổi True khi chủ động chạy toàn range.
TAKEOVER_STALE = False  # Chỉ True khi chắc chắn phiên cũ đã dừng.
if RUN_FULL_RANGE:
    if not load_semantic_scholar_key():
        raise RuntimeError('Thêm SEMANTIC_SCHOLAR_API_KEY vào Colab Secrets và bật notebook access trước khi chạy toàn range.')
    config.execution['allow_anonymous_s2'] = False
    full_ids = generate_ids(**config.range)
    print('Chạy toàn range:', full_ids[0], '→', full_ids[-1], '| Tổng:', len(full_ids))
    summary = await run_pipeline(
        config, {'arxiv', 's2'}, resume=True, limit=None,
        takeover_stale=TAKEOVER_STALE,
    )
    print(summary)
    store = ProgressStore(config.paths['persistent_root'])
    print(store.status_counts())
else:
    print('Bỏ qua toàn range: RUN_FULL_RANGE=False')


In [ ]:
from crawler.export import export_snapshot
import uuid
manifest = export_snapshot(config, 'export-' + uuid.uuid4().hex)
print(manifest)


## Resume và giới hạn phiên
Chạy lại cell smoke để tiếp tục 10 ID đầu, hoặc cell toàn range để tiếp tục toàn bộ range cấu hình. Chỉ dùng takeover_stale=True sau khi chắc chắn phiên ghi cũ đã dừng. Không chạy hai notebook/process cùng dataset.


### Checkpoint và hiệu năng
Progress gồm jobs.csv và jobs.journal.jsonl; không xóa journal hoặc dùng code cũ sau nâng cấp. Reconcile đầy đủ chạy khi bắt đầu/resume; trong phiên chỉ xác minh result vừa publish. Batch metadata theo IDvN giữ ngày từng phiên bản. Disk metrics dùng baseline + delta. Sau cập nhật, restart kernel, dùng đúng root cũ và smoke/interrupt trước full range.


## Đổi tháng và khoảng ID
Sửa range trong config/crawler.yaml của repo (trên Drive là code/config/crawler.yaml), rồi chạy lại cell load_config. start_month/end_month có dạng YYYY-MM và hiện phải cùng tháng; start_id/end_id là số nguyên, hai đầu inclusive. Đây là tháng trong arXiv ID, không phải khoảng ngày/giờ xuất bản bất kỳ. Với dataset mới, đổi paths.persistent_root sang thư mục tương đối mới; giữ root cũ khi resume cùng dataset. Xem README để có ví dụ đầy đủ.
